In [1]:
# ============================================================
# LOAN APPROVAL & RISK ANALYTICS
# File: 02_data_cleaning.ipynb
# Purpose: Clean and prepare the raw loan dataset
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

print("Libraries imported successfully!")


# ============================================================
# 2. LOAD RAW DATASET
# ============================================================

DATA_PATH = "../data/loan_train.csv"

df = pd.read_csv(DATA_PATH)

print("Raw dataset loaded successfully!")

print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")


# ============================================================
# 3. CREATE A COPY
# ============================================================

clean_df = df.copy()

print("\nWorking copy created.")


# ============================================================
# 4. CHECK DUPLICATE ROWS
# ============================================================

print("\n" + "=" * 60)
print("DUPLICATE ROW ANALYSIS")
print("=" * 60)

duplicate_count = clean_df.duplicated().sum()

print(f"Duplicate rows found: {duplicate_count}")


# ============================================================
# 5. REMOVE DUPLICATE ROWS
# ============================================================

if duplicate_count > 0:
    clean_df = clean_df.drop_duplicates()
    print(f"Removed {duplicate_count} duplicate rows.")
else:
    print("No duplicate rows found.")


# ============================================================
# 6. REMOVE LOAN_ID
# ============================================================

print("\n" + "=" * 60)
print("REMOVING UNNECESSARY COLUMNS")
print("=" * 60)

if "Loan_ID" in clean_df.columns:
    clean_df = clean_df.drop(columns=["Loan_ID"])
    print("Loan_ID removed successfully.")
else:
    print("Loan_ID column not found.")


# ============================================================
# 7. STANDARDIZE COLUMN NAMES
# ============================================================

clean_df.columns = (
    clean_df.columns
    .str.strip()
    .str.replace(" ", "_")
)

print("\nColumn names standardized.")


# ============================================================
# 8. CHECK MISSING VALUES BEFORE CLEANING
# ============================================================

print("\n" + "=" * 60)
print("MISSING VALUES BEFORE CLEANING")
print("=" * 60)

missing_before = clean_df.isnull().sum()

print(missing_before[missing_before > 0])


# ============================================================
# 9. HANDLE NUMERICAL MISSING VALUES
# ============================================================

numerical_columns = clean_df.select_dtypes(
    include=["int64", "float64"]
).columns

print("\nNumerical columns:")
print(list(numerical_columns))


for column in numerical_columns:

    if clean_df[column].isnull().sum() > 0:

        median_value = clean_df[column].median()

        clean_df[column] = clean_df[column].fillna(
            median_value
        )

        print(
            f"Filled missing values in {column} "
            f"with median: {median_value}"
        )


# ============================================================
# 10. HANDLE CATEGORICAL MISSING VALUES
# ============================================================

categorical_columns = clean_df.select_dtypes(
    include=["object"]
).columns

print("\nCategorical columns:")
print(list(categorical_columns))


for column in categorical_columns:

    if clean_df[column].isnull().sum() > 0:

        mode_value = clean_df[column].mode()[0]

        clean_df[column] = clean_df[column].fillna(
            mode_value
        )

        print(
            f"Filled missing values in {column} "
            f"with mode: {mode_value}"
        )


# ============================================================
# 11. SPECIAL CHECK FOR DEPENDENTS
# ============================================================

if "Dependents" in clean_df.columns:

    clean_df["Dependents"] = (
        clean_df["Dependents"]
        .astype(str)
        .str.strip()
    )

    clean_df["Dependents"] = clean_df[
        "Dependents"
    ].replace({
        "3+": "3+"
    })

    print("\nDependents values:")
    print(clean_df["Dependents"].value_counts())


# ============================================================
# 12. CLEAN CATEGORICAL TEXT VALUES
# ============================================================

for column in categorical_columns:

    clean_df[column] = (
        clean_df[column]
        .astype(str)
        .str.strip()
    )


print("\nCategorical text values cleaned.")


# ============================================================
# 13. CHECK MISSING VALUES AFTER CLEANING
# ============================================================

print("\n" + "=" * 60)
print("MISSING VALUES AFTER CLEANING")
print("=" * 60)

missing_after = clean_df.isnull().sum()

print(missing_after[missing_after > 0])


# ============================================================
# 14. CHECK TOTAL MISSING VALUES
# ============================================================

total_missing = clean_df.isnull().sum().sum()

print(f"\nTotal remaining missing values: {total_missing}")


# ============================================================
# 15. CHECK DUPLICATES AGAIN
# ============================================================

remaining_duplicates = clean_df.duplicated().sum()

print(
    f"Remaining duplicate rows: "
    f"{remaining_duplicates}"
)


# ============================================================
# 16. CHECK DATA TYPES
# ============================================================

print("\n" + "=" * 60)
print("DATA TYPES AFTER CLEANING")
print("=" * 60)

print(clean_df.dtypes)


# ============================================================
# 17. CHECK UNIQUE VALUES
# ============================================================

print("\n" + "=" * 60)
print("UNIQUE VALUES")
print("=" * 60)

for column in clean_df.columns:

    print(f"\n{column}:")
    print(clean_df[column].unique())


# ============================================================
# 18. BASIC STATISTICS AFTER CLEANING
# ============================================================

print("\n" + "=" * 60)
print("STATISTICAL SUMMARY AFTER CLEANING")
print("=" * 60)

print(clean_df.describe())


# ============================================================
# 19. TARGET VARIABLE CHECK
# ============================================================

print("\n" + "=" * 60)
print("LOAN STATUS AFTER CLEANING")
print("=" * 60)

print(clean_df["Loan_Status"].value_counts())

print("\nPercentage:")

print(
    clean_df["Loan_Status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


# ============================================================
# 20. FINAL DATASET SHAPE
# ============================================================

print("\n" + "=" * 60)
print("FINAL DATASET SHAPE")
print("=" * 60)

print(f"Rows    : {clean_df.shape[0]}")
print(f"Columns : {clean_df.shape[1]}")


# ============================================================
# 21. DISPLAY CLEAN DATA
# ============================================================

print("\n" + "=" * 60)
print("CLEAN DATASET - FIRST 10 ROWS")
print("=" * 60)

print(clean_df.head(10))


# ============================================================
# 22. SAVE CLEAN DATASET
# ============================================================

OUTPUT_PATH = "../data/loan_cleaned.csv"

clean_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nClean dataset saved successfully!")
print(f"Location: {OUTPUT_PATH}")


# ============================================================
# 23. CREATE CLEANING SUMMARY
# ============================================================

cleaning_summary = pd.DataFrame({
    "Metric": [
        "Original Rows",
        "Original Columns",
        "Final Rows",
        "Final Columns",
        "Duplicates Removed",
        "Remaining Missing Values"
    ],
    "Value": [
        df.shape[0],
        df.shape[1],
        clean_df.shape[0],
        clean_df.shape[1],
        duplicate_count,
        total_missing
    ]
})

print("\n" + "=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)

print(cleaning_summary)


# ============================================================
# 24. SAVE CLEANING SUMMARY
# ============================================================

cleaning_summary.to_csv(
    "../data/cleaning_summary.csv",
    index=False
)

print("\nCleaning summary saved successfully!")


# ============================================================
# 25. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("FINAL DATA VALIDATION")
print("=" * 60)

if total_missing == 0:
    print("✓ No missing values remain.")

if remaining_duplicates == 0:
    print("✓ No duplicate rows remain.")

if "Loan_Status" in clean_df.columns:
    print("✓ Target variable exists.")

print("\nDATA CLEANING COMPLETED SUCCESSFULLY!")
print("=" * 60)

Libraries imported successfully!
Raw dataset loaded successfully!
Rows    : 614
Columns : 13

Working copy created.

DUPLICATE ROW ANALYSIS
Duplicate rows found: 0
No duplicate rows found.

REMOVING UNNECESSARY COLUMNS
Loan_ID removed successfully.

Column names standardized.

MISSING VALUES BEFORE CLEANING
Gender              13
Married              3
Dependents          15
Self_Employed       32
LoanAmount          22
Loan_Amount_Term    14
Credit_History      50
dtype: int64

Numerical columns:
['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
Filled missing values in LoanAmount with median: 128.0
Filled missing values in Loan_Amount_Term with median: 360.0
Filled missing values in Credit_History with median: 1.0

Categorical columns:
['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']
Filled missing values in Gender with mode: Male
Filled missing values in Married with mode: Yes
Filled missing 

C:\Users\Princ\AppData\Local\Temp\ipykernel_21700\3357395672.py:138: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = clean_df.select_dtypes(
